In [ ]:
%load_ext autoreload
%autoreload 2

import os
import json
import polars as pl
import pandas as pd
from tqdm import tqdm

from plotnine import *
import matplotlib.pyplot as plt

## Process ProteinGym data

In [ ]:
# dms_dir = '/s/project/deeprvat/ukb_gym/experimental_assays/protein_gym/DMS_scores_SNPs_zeroshot/'
dms_dir = '/s/project/deeprvat/ukb_gym/experimental_assays/protein_gym/DMS_ProteinGym_substitutions/'

pg_list = []
for filename in tqdm(os.listdir(dms_dir)):
    if 'HUMAN' in filename:
        if filename.endswith('.csv'):
            temp = pl.read_csv(f'{dms_dir}{filename}')
            big_name = filename.split('.')[0]
            temp = temp.with_columns(
                pl.lit(big_name.split('_')[0]).alias('protein_name'),
                # pl.lit('_'.join(big_name.split('_')[2:])).alias('experiment_name'),
                pl.lit(filename.split('.')[0]).alias('file_name'),
                )
            pg_list.append(temp)

pgdf = pl.concat(pg_list)
pgdf = pgdf.with_columns(
    pl.col('protein_name').alias('gene_name'),
)
pgdf

In [ ]:
plt.hist(pgdf.filter(pl.col('DMS_score')>0)['DMS_score'], bins=100, log=True)
plt.show()

In [ ]:
plt.hist(pgdf.filter(pl.col('DMS_score')<0)['DMS_score'], bins=100, log=True)
plt.show()

In [ ]:
pgdf['protein_name'].n_unique()

In [ ]:
gene_df = pl.read_parquet('/s/project/deeprvat/deeprvat_input/protein_coding_genes.parquet')
gene_df = gene_df.with_columns(
    pl.col('gene').str.split('.').list.get(0).alias('gene_id'),
    pl.col('gene').str.split('.').list.get(0).alias('region')
)
gene_df = gene_df.drop(['gene', '__index_level_0__', 'id'])
gene_df

In [ ]:
set(pgdf['protein_name'].unique().to_list()) - set(gene_df['gene_name'].unique().to_list())

In [ ]:
protname_dict = {
    'A4': 'APP',
    'B2L11': 'BCL2L11',
    'CAR11': 'CARD11',
    'CBPA2': 'CPA2',
    'CP2C9': 'CYP2C9',
    'DNJA1': 'DNAJA1',
    'GDIA': 'GDI1',
    'GLPA': 'GYPA',
    'HECD1': 'HECTD1',
    'HEM3': 'HMBS',
    'HMDH': 'HMGCR',
    'HXK4': 'GCK',
    'LYAM1': 'SELL',
    'MK01': 'MAPK1',
    'MTHR': 'MTHFR',
    'NKX31': 'NKX3-1',
    'NUD15': 'NUDT15',
    'OPSD': 'RHO',
    'OTU7A': 'OTUD7A',
    'P53': 'TP53',
    'PAI1': 'SERPINE1',
    'PR40A': 'PRPF40A',
    'Q53Z42': 'HLA-A',
    'RASH': 'HRAS',
    'RASK': 'KRAS',
    'RD23A': 'RAD23A',
    'S22A1': 'SLC22A1',
    'SC6A4': 'SLC6A4',
    'SERC': 'PSAT1',
    'SRBS1': 'SORBS1',
    'SYUA': 'SNCA',
    'TADBP': 'TARDBP',
    'TPOR': 'THPO',
    'UBC9': 'UBE2I',
    'VKOR1': 'VKORC1',
}


# Convert to pandas
df_pd = pgdf.filter(pl.col('protein_name').is_in(protname_dict.keys())).to_pandas()

# Map only if key in map_dict, else keep original value
df_pd['gene_name'] = df_pd['protein_name'].apply(
    lambda x: protname_dict[x]
)

df_pd['gene_name'].value_counts().sort_values(ascending=False)

In [ ]:
# Append back to protein_gym
sub_pgdf = pgdf.filter(~pl.col('protein_name').is_in(protname_dict.keys()))

pgdf = pl.concat([sub_pgdf, pl.from_pandas(df_pd)])
pgdf

In [ ]:
# dms_df = pgdf[['gene_name', 'mutant', 'DMS_score']].join(gene_df, on='gene_name')
dms_df = pgdf.join(gene_df, on='gene_name')
print(dms_df['gene_name'].n_unique())
dms_df

In [ ]:
dms_df.columns = [colname.lower() for colname in dms_df.columns]
dms_df

In [ ]:
# dms_df.write_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/protein_gym/250717_proteingym_SNP_DMS_scores_human_coding_genes.parquet')

## DMS score consistency

In [ ]:
pgdf = pl.read_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/protein_gym/250717_proteingym_SNP_DMS_scores_human_coding_genes.parquet')

pgdf['gene_name'].value_counts().sort('count', descending=True).head(10)

In [ ]:
a = pgdf.group_by('file_name').agg(
    pl.col('dms_score').mean().alias('dms_score_mean'),
    pl.col('dms_score').median().alias('dms_score_median'),
    pl.col('dms_score').std().alias('dms_score_se'),
)

plt_df = pgdf.join(a, on='file_name').sort('dms_score_median', descending=True)
plt_df

In [ ]:
(
    ggplot(plt_df, aes(x='file_name', y='dms_score')) +
    geom_boxplot() +
    scale_y_log10() +
    theme_bw() +
    theme(
        axis_text_x=element_text(rotation=90),
        figure_size=(12, 8)
    )
)

In [ ]:
dup_genes = plt_df[['file_name', 'gene_name']].unique().filter(pl.col('gene_name').is_duplicated())
dup_genes

In [ ]:
pgdf.filter(~pl.col('gene_name').is_in(dup_genes['gene_name']))['gene_name'].value_counts().sort('count', descending=True).head(10)

In [ ]:
(
    ggplot(plt_df.filter(pl.col('gene_name').is_in(dup_genes['gene_name'])), aes(x='gene_name', y='dms_score', fill='file_name')) +
    geom_boxplot() +
    theme_bw() +
    # facet_wrap('gene_name', scales='free') +
    theme(
        axis_text_x=element_text(rotation=90),
        figure_size=(12, 6),
        legend_position='none'
    )
)

In [ ]:
dup_plt = plt_df.filter(pl.col('gene_name')=='PTEN').pivot(
    index='mutant',
    columns='file_name',
    values='dms_score'
).drop_nulls()


(
    ggplot(dup_plt, aes(x=dup_plt.columns[1], y=dup_plt.columns[2])) +
    geom_bin_2d(bins=100) +
    geom_abline(color='red') +
    theme_bw()
)

In [ ]:
dup_plt = plt_df.filter(pl.col('gene_name')=='TP53').pivot(
    index='mutant',
    columns='file_name',
    values='dms_score'
).drop_nulls()

print(dup_plt.median())
dup_plt

In [ ]:
(
    ggplot(dup_plt, aes(x='P53_HUMAN_Kotler_2018', y='P53_HUMAN_Giacomelli_2018_Null_Etoposide')) +
    geom_bin_2d(bins=100) +
    geom_abline(color='red') +
    theme_bw()
)

In [ ]:
(
    ggplot(pgdf.filter(pl.col('gene_name')=='BRCA1'), aes(x='dms_score')) +
    geom_histogram(bins=100) +
    theme_bw()
)

In [ ]:
(
    ggplot(pgdf.filter(pl.col('gene_name')=='GCK'), aes(x='dms_score')) +
    geom_histogram(bins=100) +
    theme_bw()
)

In [ ]:
dup_plt = plt_df.filter(pl.col('gene_name')=='GCK').pivot(
    index='mutant',
    columns='file_name',
    values='dms_score'
).drop_nulls()


(
    ggplot(dup_plt, aes(x=dup_plt.columns[1], y=dup_plt.columns[2])) +
    geom_bin_2d(bins=100) +
    geom_abline(color='red') +
    theme_bw()
)

In [ ]:
pgdf = pgdf.with_columns(
    (pl.col('gene_name') + '_' + pl.col('mutant')).alias('gene_mutant')
)

reps = pgdf['gene_mutant'].value_counts().filter(pl.col('count')>1).sort("count", descending=True)
reps

In [ ]:
pg_reps = pgdf.filter(pl.col('gene_mutant').is_in(reps['gene_mutant']))

pg_reps_stats = (
    pgdf.group_by('gene_mutant')
    .agg([
        pl.count('dms_score').alias('num_reps'),
        pl.mean('dms_score').alias('dms_score_mean'),
        pl.std('dms_score').alias('dms_score_std'),
        pl.min('dms_score').alias('dms_score_min'),
        pl.max('dms_score').alias('dms_score_max')
    ])
).sort('dms_score_std', descending=True)

pg_reps_stats

In [ ]:
plt_df = pg_reps_stats.filter(~pl.col('dms_score_std').is_null())#.filter(pl.col('dms_score_std')<100)


(
    ggplot(plt_df, aes(x='dms_score_mean', y='dms_score_std')) +
    geom_bin2d(bins=100) +
    geom_abline() +
    scale_x_log10() +
    scale_y_log10() +
    theme_bw()
)

## Intersection of genes with genebass

In [ ]:
gb_res = pd.read_parquet('/s/project/deeprvat/ukb_gym/genebass/genebass_all_associations_p_e-5.parquet')

pval_cutoffs = {'burden': 6.7e-7, 'skato': 2.5e-7} # pvalue thresholds used in genebass paper
mask = (gb_res['Pvalue'] < pval_cutoffs['skato'])
mask |= (gb_res['Pvalue_Burden'] < pval_cutoffs['burden'])
gb_res["significant"] = mask

gb_res = gb_res.query("(significant == True) & (modifier != 'custom') & ('pLoF' in annotation)")
gb_res

In [ ]:
dg_genes = dms_df.filter(pl.col('gene_id').is_in(gb_res['gene_id'].unique()))
dg_genes

## Check intersection of variants

In [ ]:
gb_rap = pl.read_parquet('/s/project/deeprvat/ukb_gym/genebass/250709_rap_prs_phenos_genebass_assocs.parquet')
gb_rap

In [ ]:
rap_vars = pl.read_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass_1e6_coding_variants.parquet')
rap_vars = rap_vars.filter(pl.col('AF_ukb')<0.001)

rap_vars = rap_vars.filter(pl.col('region').is_in(gb_rap['gene_id'].unique()))
# rap_vars = rap_vars.filter(pl.col('region').is_in(gb_res['gene_id'].unique()))
rap_vars

In [ ]:
mj = rap_vars[['region', 'mutant']].unique().join(dms_df, on=['region', 'mutant'], how='inner')
mj

In [ ]:
vc = mj['gene_name'].value_counts().sort('count', descending=True)
print(vc['count'].sum())
vc

In [ ]:
# Example counts
vc = (
    mj['gene_name']
    .value_counts()
    .sort('count', descending=True)
).to_pandas()

# Make gene_name a categorical with sorted levels
vc['gene_name'] = pd.Categorical(vc['gene_name'], categories=vc['gene_name'], ordered=True)

# Plot
(
    ggplot(vc, aes(x='gene_name', y='count')) +
    geom_col() +
    theme_bw() +
    scale_y_log10() +
    annotation_logticks(sides='l') +
    theme(
        axis_text_x=element_text(angle=90),
        figure_size=(4, 3)
    )
)